# NB5 — SMPL 하체 depth-fit (β 고정, λ_prior = 0) **[GPU]**
| 단계 | 스크립트 | 산출 (`sessions/{SESSION}/04_fit/`) |
|---|---|---|
| A0 root 정렬 | `a0_align_v2.py` | `A0/A0_pose_per_frame_v2.npz`, `A0/overlay_ply/` |
| near(오른) 다리 9-DOF | `depthfit_near_leg_lam0.py --stage f3` | `near/depthfit_near_leg_lam0_f3.npz/.png` |
| far(왼) 다리 9-DOF | `depthfit_far_leg_lam0.py --stage far2` | `far/depthfit_far_leg_lam0_far2.npz/.png` |
| (선택) 발목 refit | `08_ankle_refit.py` | `ankle/depthfit_ankle_a1.npz` |

불변 (4/20 과 동일): β = STAND ii13 `rawdepth_w05_leftcorr` freeze, body_pose init = 4D-Humans, Huber δ 3cm, near = 오른다리 (카메라 쪽).

In [ ]:
#@title 0. 설정 — 세션마다 여기만 바꾸면 됨
ROOT    = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION = "0729_full_3kmh"                        #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
os.environ["WALK_ROOT"] = ROOT
os.environ["WALK_SESSION"] = SESSION
sys.path.insert(0, f"{ROOT}/scripts")
import walkcfg; importlib.reload(walkcfg)
assert os.path.isdir(f"{ROOT}/scripts"), f"{ROOT}/scripts 없음 — colab_kit 업로드 위치 확인"
print("세션 폴더:", walkcfg.SESS)

In [ ]:
#@title 1. 의존성 + 입력 확인
!pip -q install smplx
import torch, os
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU — 매우 느림)")
for k in ("FROZEN_NPZ", "SMPL_MODEL", "CAM_INTR", "FOURDH_SEQ", "DEPTH_SEQ", "VIDEO_SEQ"):
    p = getattr(walkcfg, k); print(f"{k:11s} {'OK ' if os.path.exists(p) else '없음'} {p}")
import glob
print("재-anchor depth:", len(glob.glob(f"{walkcfg.DEPTH_SEQ}/WALK_side_*_person_metric.npy")), "| .bak:", len(glob.glob(f"{walkcfg.DEPTH_SEQ}/*.bak")))

In [ ]:
#@title 2. A0 — root(global_orient, transl) 를 depth 에 정렬
!cd "{ROOT}/scripts" && python a0_align_v2.py 2>&1 | tail -40

In [ ]:
#@title 3. near(오른) 다리 — 전 프레임 (f3)
!cd "{ROOT}/scripts" && python depthfit_near_leg_lam0.py --stage f3 2>&1 | tail -30
from IPython.display import Image, display
display(Image(f"{walkcfg.NEAR_DIR}/depthfit_near_leg_lam0_f3.png"))

In [ ]:
#@title 4. far(왼) 다리 — near θ 주입 고정 (far2)
!cd "{ROOT}/scripts" && python depthfit_far_leg_lam0.py --stage far2 2>&1 | tail -30
display(Image(f"{walkcfg.FAR_DIR}/depthfit_far_leg_lam0_far2.png"))

In [ ]:
#@title 5. 점검 — far 측정 비율 (7/15 z-sign 의문 관련)
import numpy as np
f = np.load(walkcfg.FAR_F2, allow_pickle=True)
lab = f["label"].astype(str); u, c = np.unique(lab, return_counts=True)
print(dict(zip(u, c)))
meas = (lab == "MEASURED").mean()
print(f"far MEASURED 비율 = {meas*100:.1f}%   (4/20: 84%)")
if meas < 0.6:
    print("⚠ far 측정 비율이 낮음 — near/far z-ordering 부호 의심 (7/15 shoes_shell S4 _zminus 결과 참고). 진행 전 확인 필요.")

In [ ]:
#@title 6. (선택) 발목 refit
RUN_ANKLE = False  #@param {type:"boolean"}
if RUN_ANKLE:
    !cd "{ROOT}/scripts" && python 08_ankle_refit.py 2>&1 | tail -20

## 시각화 (NB5)

In [ ]:
#@title 시각화 — NB5 결과 확인 (viz/NB5/  + viz/videos/)
FIDS = "5660 5750"   #@param {type:"string"}
N_EVEN = 0           #@param {type:"integer"}
# 셀 0 을 안 돌렸어도 혼자 동작하도록 (드라이브 마운트 + 세션 설정)
import os, sys, importlib
if "walkcfg" not in globals():
    ROOT = "/content/drive/MyDrive/walking_0729"   # 셀 0 과 같게
    SESSION = "0729_full_3kmh"                     # 셀 0 과 같게
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WALK_ROOT"] = ROOT; os.environ["WALK_SESSION"] = SESSION
    sys.path.insert(0, f"{ROOT}/scripts"); import walkcfg
print("세션:", walkcfg.SESS)
MAKE_VIDEO = True    #@param {type:"boolean"}
FPS = 15             #@param {type:"number"}
!pip -q install smplx imageio-ffmpeg
!cd "{ROOT}/scripts" && python viz_stages.py --nb 5 --fids {FIDS} --n {N_EVEN}
if MAKE_VIDEO:
    !cd "{ROOT}/scripts" && python viz_video.py --nb 5 --fps {FPS}
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob(f"{walkcfg.SESS}/viz/NB5/*.png")):
    print(os.path.basename(p)); display(Image(p, width=1100))
from IPython.display import Video
_mp4 = f"{walkcfg.SESS}/viz/videos/NB5_fit.mp4"
if os.path.exists(_mp4):
    print("영상:", _mp4, f"{os.path.getsize(_mp4)/1e6:.1f} MB  (노트북에 넣어 재생 — 확인 후 출력 지우기 권장)")
    display(Video(_mp4, embed=True, width=1100))